In [2]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy

In [3]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.ones_like(priors)
    return priors / np.sum(priors)

In [4]:
def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + (i*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i)*1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [5]:
def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [6]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.0
    for partition in partitions:
        acc_loss_p  = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [7]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [8]:
def display_tree(tree, n):
    print("Optimal")
    for acc_loss, partition in tree[:n]:
        print(f"  ({acc_loss:.4e}, {partition})")

def find_partitions_optimal(X, thresholds, priors, threshold_true, c, display=None, n=5):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))
    best_partition, best_loss = None, np.inf
    tree = []
    for partitions in partitions_set:
        acc_loss = evaluate_system(X, partitions, thresholds, priors, threshold_true, c)
        tree.append((acc_loss, partitions))
        if acc_loss < best_loss:
            best_loss      = acc_loss
            best_partition = partitions

    if display:
        display_tree(sorted(tree, key=lambda x: x[0]), n)
    return best_partition

In [9]:
def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({-acc_loss:.4e}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)

def find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=False, eps=1e-9):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1
    
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)

    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c) * np.sum(priors[a])
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c) * np.sum(priors[b])

        acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab]) * eps
        acc_loss_a_eps = evaluate_partition(X_eps, a, thresholds, priors, threshold_true, c) * np.sum(priors[a]) * eps
        acc_loss_b_eps = evaluate_partition(X_eps, b, thresholds, priors, threshold_true, c) * np.sum(priors[b]) * eps

        acc_loss_ab += acc_loss_ab_eps
        acc_loss_a += acc_loss_a_eps
        acc_loss_b += acc_loss_b_eps
        
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        if display:
            display_priority_queue(pq, P)
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        ab = sorted(a + b)
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)

        acc_loss_a_eps = evaluate_partition(X_eps, a, thresholds, priors, threshold_true, c) * eps
        acc_loss_b_eps = evaluate_partition(X_eps, b, thresholds, priors, threshold_true, c) * eps
        acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * eps

        acc_loss_a += acc_loss_a_eps
        acc_loss_b += acc_loss_b_eps
        acc_loss_ab += acc_loss_ab_eps

        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]

            pq2 = []
            for acc_loss, (x_id, y_id) in pq:
                if x_id not in {a_id, b_id} and y_id not in {a_id, b_id}:
                    heapq.heappush(pq2, (acc_loss, (x_id, y_id)))
            pq = deepcopy(pq2)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    p = P[p_id]
                    merged = sorted(ab + p)
                    acc_loss_merged = evaluate_partition(X, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged])
                    acc_loss_p = evaluate_partition(X, p, thresholds, priors, threshold_true, c) * np.sum(priors[p])
                    acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])

                    acc_loss_merged_eps = evaluate_partition(X_eps, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged]) * eps
                    acc_loss_p_eps = evaluate_partition(X_eps, p, thresholds, priors, threshold_true, c) * np.sum(priors[p]) * eps
                    acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab]) * eps

                    acc_loss_merged += acc_loss_merged_eps
                    acc_loss_p += acc_loss_p_eps
                    acc_loss_ab += acc_loss_ab_eps

                    gain = -(acc_loss_p + acc_loss_ab - acc_loss_merged)
                    heapq.heappush(pq, (gain, (new_id, p_id)))
    return list(P.values())

In [10]:
def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [11]:
def manip_thresh_023(priors, c):
    p = priors[[0,2,3]]
    q = normalize_priors(p)
    q0 = q[0]
    q01 = q[0] + q[1]
    # Jump to 0.6: p01 - c*(0.6 - x) > p0 => x > 0.6 - (p01 - p0)/c
    # Jump to 1.0: 1.0 - c*(1.0 - x) > p0 => x > 1.0 - (1.0 - p0)/c
    thresh_2 = 0.6 - (q01 - q0) / c
    thresh_3 = 1.0 - (1.0 - q0) / c
    return min(thresh_2, thresh_3)

In [12]:
def merge_gain_01(X, thresholds, priors, threshold_true, c):
    loss_0 = evaluate_partition(X, [0], thresholds, priors, threshold_true, c) * priors[0]
    loss_1 = evaluate_partition(X, [1], thresholds, priors, threshold_true, c) * priors[1]
    loss_01 = evaluate_partition(X, [0,1], thresholds, priors, threshold_true, c) * np.sum(priors[[0,1]])
    return loss_0 + loss_1 - loss_01

In [13]:
X = np.arange(0, 1.001, 0.001).round(5)

In [14]:
thresholds = np.array([0., 0.2, 0.6, 1.0])
c = 0.75

### Experiment 1

Fix p0 and p2, vary p1, and assign p3 the remaining

In [15]:
results_1 = {"p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}
for p1 in [0.001, 0.002, 0.003, 0.005, 0.007, 0.010, 0.012, 0.014, 0.016, 0.020, 0.030, 0.050, 0.070]:
    p0, p2 = 0.32, 0.36
    p3 = round(1.0 - p0 - p1 - p2, 5)
    priors = np.array([p0, p1, p2, p3])
    
    mt = manip_thresh_023(priors, c)
    tt = mt 
    
    gain = merge_gain_01(X, thresholds, priors, tt, c)
    partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results_1["p0"].append(p0)
    results_1["p1"].append(p1)
    results_1["p2"].append(p2)
    results_1["p3"].append(p3)
    results_1["tt"].append(tt)
    results_1["partition_opt"].append(partition_opt)
    results_1["partition_greedy"].append(partition_greedy)
    results_1["loss_opt"].append(loss_opt)
    results_1["loss_greedy"].append(loss_greedy)
    results_1["r"].append(r)

In [16]:
df_results_1 = pd.DataFrame(results_1)
df_results_1

,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
0,0.32,0.001,0.36,0.319,0.093760,"[[0, 1, 2, 3]]","[[0, 1, 2, 3]]",0.030050,0.030050,1.000000
1,0.32,0.002,0.36,0.318,0.094188,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030559,0.030559,1.000000
2,0.32,0.003,0.36,0.317,0.094617,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.030654,0.031049,1.012873
3,0.32,0.005,0.36,0.315,0.095477,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.031169,0.032048,1.028205
4,0.32,0.007,0.36,0.313,0.096341,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.031687,0.033047,1.042908
5,0.32,0.010,0.36,0.310,0.097643,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.032308,0.034046,1.053803
6,0.32,0.012,0.36,0.308,0.098516,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.032835,0.035045,1.067300
7,0.32,0.014,0.36,0.306,0.099391,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.033367,0.036044,1.080240
8,0.32,0.016,0.36,0.304,0.100271,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.033902,0.037043,1.092645
9,0.32,0.020,0.36,0.300,0.102041,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.034985,0.101219,2.893204


### Experiment 2

Fix p0 and p3, vary p1, and assign p2 the remaining

In [17]:
results_2 = {"p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}
for p1 in [0.001, 0.002, 0.003, 0.005, 0.007, 0.010, 0.012, 0.014, 0.016, 0.020, 0.030, 0.050, 0.070]:
    p0, p3 = 0.32, 0.3
    p2 = round(1.0 - p0 - p1 - p3, 5)
    priors = np.array([p0, p1, p2, p3])
    
    mt = manip_thresh_023(priors, c)
    tt = mt
    
    gain = merge_gain_01(X, thresholds, priors, tt, c)
    partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results_2["p0"].append(p0)
    results_2["p1"].append(p1)
    results_2["p2"].append(p2)
    results_2["p3"].append(p3)
    results_2["tt"].append(tt)
    results_2["partition_opt"].append(partition_opt)
    results_2["partition_greedy"].append(partition_greedy)
    results_2["loss_opt"].append(loss_opt)
    results_2["loss_greedy"].append(loss_greedy)
    results_2["r"].append(r)

In [18]:
df_results_2 = pd.DataFrame(results_2)
df_results_2

,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
0,0.32,0.001,0.379,0.3,0.093760,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030144,0.030144,1.000000
1,0.32,0.002,0.378,0.3,0.094188,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030559,0.030559,1.000000
2,0.32,0.003,0.377,0.3,0.094617,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.030654,0.094899,3.095780
3,0.32,0.005,0.375,0.3,0.095477,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.031169,0.095844,3.075000
4,0.32,0.007,0.373,0.3,0.096341,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.031687,0.096749,3.053249
5,0.32,0.010,0.370,0.3,0.097643,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.032308,0.097542,3.019171
6,0.32,0.012,0.368,0.3,0.098516,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.032835,0.098350,2.995254
7,0.32,0.014,0.366,0.3,0.099391,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.033367,0.099131,2.970958
8,0.32,0.016,0.364,0.3,0.100271,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.033902,0.099860,2.945545
9,0.32,0.020,0.360,0.3,0.102041,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.034985,0.101219,2.893204


### Experiment 3

Fix p2 and p3, vary p1, and assign p0 the remaining

In [19]:
results_3 = {"p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}
for p1 in [0.001, 0.002, 0.003, 0.005, 0.007, 0.010, 0.012, 0.014, 0.016, 0.020, 0.030, 0.050, 0.070]:
    p2, p3 = 0.36, 0.3
    p0 = round(1.0 - p1 - p2 - p3, 5)
    priors = np.array([p0, p1, p2, p3])
    
    mt = manip_thresh_023(priors, c)
    tt = mt
    
    gain = merge_gain_01(X, thresholds, priors, tt, c)
    partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results_3["p0"].append(p0)
    results_3["p1"].append(p1)
    results_3["p2"].append(p2)
    results_3["p3"].append(p3)
    results_3["tt"].append(tt)
    results_3["partition_opt"].append(partition_opt)
    results_3["partition_greedy"].append(partition_greedy)
    results_3["loss_opt"].append(loss_opt)
    results_3["loss_greedy"].append(loss_greedy)
    results_3["r"].append(r)

In [20]:
df_results_3 = pd.DataFrame(results_3)
df_results_3

,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
0,0.339,0.001,0.36,0.3,0.119119,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.040759,0.119837,2.940123
1,0.338,0.002,0.36,0.3,0.118236,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.040420,0.118791,2.938952
2,0.337,0.003,0.36,0.3,0.117352,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.040080,0.117741,2.937662
3,0.335,0.005,0.36,0.3,0.115578,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.039401,0.115629,2.934711
4,0.333,0.007,0.36,0.3,0.113797,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.038721,0.113501,2.931244
5,0.330,0.010,0.36,0.3,0.111111,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.038042,0.111259,2.924632
6,0.328,0.012,0.36,0.3,0.109312,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.037363,0.109087,2.919679
7,0.326,0.014,0.36,0.3,0.107505,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.036683,0.106913,2.914488
8,0.324,0.016,0.36,0.3,0.105691,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.036004,0.104711,2.908324
9,0.320,0.020,0.36,0.3,0.102041,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.034985,0.101219,2.893204


[Exp 2] Fixing p0 and p3, and varying p1 is able to acheive r > 3. 

In [21]:
df_results_2

,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
0,0.32,0.001,0.379,0.3,0.093760,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030144,0.030144,1.000000
1,0.32,0.002,0.378,0.3,0.094188,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030559,0.030559,1.000000
2,0.32,0.003,0.377,0.3,0.094617,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.030654,0.094899,3.095780
3,0.32,0.005,0.375,0.3,0.095477,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.031169,0.095844,3.075000
4,0.32,0.007,0.373,0.3,0.096341,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.031687,0.096749,3.053249
5,0.32,0.010,0.370,0.3,0.097643,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.032308,0.097542,3.019171
6,0.32,0.012,0.368,0.3,0.098516,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.032835,0.098350,2.995254
7,0.32,0.014,0.366,0.3,0.099391,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.033367,0.099131,2.970958
8,0.32,0.016,0.364,0.3,0.100271,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.033902,0.099860,2.945545
9,0.32,0.020,0.360,0.3,0.102041,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.034985,0.101219,2.893204


If p1 is too small greedy finds the optimal, and if p1 is too big, then greedy again finds the optimal. So there is a critical point for p1, where it makes sense for greedy to merge 0 and 1.

### Experiment 4

Fix p0, vary p1 and p2, and assign p3 the remaining

In [22]:
results_4 = {"p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}

p0 = 0.32
for p1 in np.arange(0.002, 0.022, 0.004):
    for p2 in np.arange(0.35, 0.40, 0.01):
        p3 = round(1.0 - p0 - p1 - p2, 3)
        priors = np.array([p0, p1, p2, p3])
        
        mt = manip_thresh_023(priors, c)
        tt  = mt

        partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
        partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
        r = approximation_ratio(loss_opt, loss_greedy)
    
        results_4["p0"].append(p0)
        results_4["p1"].append(p1)
        results_4["p2"].append(p2)
        results_4["p3"].append(p3)
        results_4["tt"].append(tt)
        results_4["partition_opt"].append(partition_opt)
        results_4["partition_greedy"].append(partition_greedy)
        results_4["loss_opt"].append(loss_opt)
        results_4["loss_greedy"].append(loss_greedy)
        results_4["r"].append(r)

In [23]:
df_results_4 = pd.DataFrame(results_4)
df_results_4.style.highlight_max("r", color="orange")

,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
0,0.320000,0.002000,0.350000,0.328000,0.094188,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030559,0.030559,1.000000
1,0.320000,0.002000,0.360000,0.318000,0.094188,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030559,0.030559,1.000000
2,0.320000,0.002000,0.370000,0.308000,0.094188,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.030559,0.030559,1.000000
3,0.320000,0.002000,0.380000,0.298000,0.092318,"[[0], [1], [2, 3]]","[[0, 2], [1, 3]]",0.092907,0.092907,1.000000
4,0.320000,0.002000,0.390000,0.288000,0.078958,"[[0, 2], [1, 3]]","[[0, 2], [1, 3]]",0.078921,0.078921,1.000000
5,0.320000,0.002000,0.400000,0.278000,0.065598,"[[0], [1, 2, 3]]","[[0, 2], [1, 3]]",0.065934,0.065934,1.000000
6,0.320000,0.006000,0.350000,0.324000,0.095909,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.031265,0.032048,1.025051
7,0.320000,0.006000,0.360000,0.314000,0.095909,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.031265,0.032048,1.025051
8,0.320000,0.006000,0.370000,0.304000,0.095909,"[[1], [0, 2, 3]]","[[0, 1, 2, 3]]",0.031265,0.032048,1.025051
9,0.320000,0.006000,0.380000,0.294000,0.090275,"[[0, 1], [2, 3]]","[[0, 1], [2, 3]]",0.090873,0.090873,1.000000


### Experiment 5

Fix p3, vary p1 and p2, and assign p0 the remaining

In [24]:
results_5 = {"p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}

p3 = 0.3
for p1 in np.arange(0.002, 0.022, 0.004):
    for p2 in np.arange(0.35, 0.40, 0.01):
        p0 = round(1.0 - p1 - p2 - p3, 3)
        priors = np.array([p0, p1, p2, p3])
        
        mt = manip_thresh_023(priors, c)
        tt  = mt

        partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
        partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
        r = approximation_ratio(loss_opt, loss_greedy)
    
        results_5["p0"].append(p0)
        results_5["p1"].append(p1)
        results_5["p2"].append(p2)
        results_5["p3"].append(p3)
        results_5["tt"].append(tt)
        results_5["partition_opt"].append(partition_opt)
        results_5["partition_greedy"].append(partition_greedy)
        results_5["loss_opt"].append(loss_opt)
        results_5["loss_greedy"].append(loss_greedy)
        results_5["r"].append(r)

In [25]:
df_results_5 = pd.DataFrame(results_5)
df_results_5.style.highlight_max("r", color="orange")

,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
0,0.348000,0.002000,0.350000,0.300000,0.131597,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.046154,0.131726,2.854069
1,0.338000,0.002000,0.360000,0.300000,0.118236,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.040420,0.118791,2.938952
2,0.328000,0.002000,0.370000,0.300000,0.104876,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.034615,0.104859,3.029264
3,0.318000,0.002000,0.380000,0.300000,0.091516,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.029411,0.029411,1.000000
4,0.308000,0.002000,0.390000,0.300000,0.078156,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.024466,0.024466,1.000000
5,0.298000,0.002000,0.400000,0.300000,0.064796,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.019481,0.019481,1.000000
6,0.344000,0.006000,0.350000,0.300000,0.128102,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.045105,0.128392,2.846512
7,0.334000,0.006000,0.360000,0.300000,0.114688,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.039061,0.114567,2.933043
8,0.324000,0.006000,0.370000,0.300000,0.101274,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.033626,0.101730,3.025312
9,0.314000,0.006000,0.380000,0.300000,0.087860,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.028132,0.028132,1.000000


### Experiment 6

Fix p0=0.292 and p2=0.39, vary p1, and assign remaining to p3

In [26]:
results_6 = {"p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}
p0, p2 = 0.292, 0.39

for p1 in [0.001, 0.005, 0.010, 0.015, 0.018, 0.02, 0.022, 0.025]:
    p3 = round(1.0 - p0 - p1 - p2, 5)
    priors = np.array([p0, p1, p2, p3])
    
    mt = manip_thresh_023(priors, c)
    tt  = mt

    partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
    r = approximation_ratio(loss_opt, loss_greedy)

    results_6["p0"].append(p0)
    results_6["p1"].append(p1)
    results_6["p2"].append(p2)
    results_6["p3"].append(p3)
    results_6["tt"].append(tt)
    results_6["partition_opt"].append(partition_opt)
    results_6["partition_greedy"].append(partition_greedy)
    results_6["loss_opt"].append(loss_opt)
    results_6["loss_greedy"].append(loss_greedy)
    results_6["r"].append(r)

In [27]:
df_results_6 = pd.DataFrame(results_6)
df_results_6.style.highlight_max("r", color="orange")

,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
0,0.292000,0.001000,0.390000,0.317000,0.056390,"[[0, 1, 2, 3]]","[[0, 1, 2, 3]]",0.016627,0.016627,1.000000
1,0.292000,0.005000,0.390000,0.313000,0.057956,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.017209,0.017209,1.000000
2,0.292000,0.010000,0.390000,0.308000,0.059933,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.018102,0.018102,1.000000
3,0.292000,0.015000,0.390000,0.303000,0.061929,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.019015,0.019015,1.000000
4,0.292000,0.018000,0.390000,0.300000,0.063136,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.019820,0.063846,3.221270
5,0.292000,0.020000,0.390000,0.298000,0.063946,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.019948,0.063676,3.192107
6,0.292000,0.022000,0.390000,0.296000,0.064758,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.020390,0.064430,3.159922
7,0.292000,0.025000,0.390000,0.293000,0.065983,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.020901,0.065010,3.110362


In [28]:
priors_max = np.array([0.292, 0.018, 0.39, 0.3])
tt = manip_thresh_023(priors_max, c)

partition_opt = find_partitions_optimal(X, thresholds, priors_max, tt, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors_max, tt, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors_max, tt, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors_max, tt, c)
r = approximation_ratio(loss_opt, loss_greedy)
r

np.float64(3.2212701612903225)

In [29]:
a, b = [1], [2]

acc_loss_a = evaluate_partition(X, a, thresholds, priors_max, tt, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors_max, tt, c)

lhs = acc_loss_a * np.sum(priors_max[a]) + acc_loss_b * np.sum(priors_max[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors_max, tt, c)
rhs = acc_loss_ab * np.sum(priors_max[ab])
gain = lhs - rhs

print(f"    threshold true: {tt:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"conditional loss a: {acc_loss_a:.7f}")
print(f"conditional loss b: {acc_loss_b:.7f}")
print(f"            loss a: {acc_loss_a * np.sum(priors_max[a]):.7f}")
print(f"            loss b: {acc_loss_b * np.sum(priors_max[b]):.7f}")
print(f"           loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"              gain: {gain}")
print(f"            merge?: {lhs - rhs > -1e-6}\n")

partition_opt = find_partitions_optimal(X, thresholds, priors_max, tt, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors_max, tt, c, display=False)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors_max, tt, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors_max, tt, c)
r = approximation_ratio(loss_opt, loss_greedy)
r

    threshold true: 0.0631
                 c: 0.75
                 a: [1]
                 b: [2]
conditional loss a: 0.0639361
conditional loss b: 0.0639361
            loss a: 0.0011508
            loss b: 0.0249351
           loss ab: 0.0639361
               LHS: 0.0260859
               RHS: 0.0260859
              gain: 0.0
            merge?: True



np.float64(3.2212701612903225)

### Experiment 6

Fix p0=0.25, vary p1, assign remaining to p2 and p3

In [30]:
results_7 = {"c": [], "p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}

c = 0.75
p0 = 0.25

for p1 in np.arange(0.001, 0.05, 0.004):
    remaining = 1 - p0 - p1

    p2_base = remaining
    p3_base = 0.

    alphas = np.arange(0, remaining, remaining*1e-2).round(2)
    for alpha in alphas:
        p2 = p2_base - alpha
        p3 = p3_base + alpha

        priors = np.array([p0, p1, p2, p3])
            
        mt = manip_thresh_023(priors, c)
        tt  = max(0,mt)

        partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
        partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
        r = approximation_ratio(loss_opt, loss_greedy)

        results_7["c"].append(c)
        results_7["p0"].append(p0)
        results_7["p1"].append(p1)
        results_7["p2"].append(p2)
        results_7["p3"].append(p3)
        results_7["tt"].append(tt)
        results_7["partition_opt"].append(partition_opt)
        results_7["partition_greedy"].append(partition_greedy)
        results_7["loss_opt"].append(loss_opt)
        results_7["loss_greedy"].append(loss_greedy)
        results_7["r"].append(r)

In [31]:
df_results_7 = pd.DataFrame(results_7).dropna()
df_results_7.iloc[[df_results_7["r"].argmax()]]
# df_results_7

,c,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_opt,loss_greedy,r
1042,0.75,0.25,0.041,0.409,0.3,0.014251,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.004361,0.014453,3.314318


In [32]:
priors_max = np.array([0.3, 0.018, 0.398, 0.3])
tt = manip_thresh_023(priors_max, c)

partition_opt = find_partitions_optimal(X, thresholds, priors_max, tt, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors_max, tt, c, display=False)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors_max, tt, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors_max, tt, c)
r = approximation_ratio(loss_opt, loss_greedy)

print(f"c              : {c:.6f}")
print(f"True Threshold : {tt:.6f}")
print("Greedy\n------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}\n")
print("Optimal\n-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}\n")
print(f"r (mult)       : {r}")

c              : 0.750000
True Threshold : 0.067468
Greedy
------
Partition      : [[0, 1], [2, 3]]
Accuracy loss  : 0.068821

Optimal
-------
Partition      : [[1], [0, 2, 3]]
Accuracy loss  : 0.021602

r (mult)       : 3.185812060673326


In [33]:
c = 0.99
p1 = 0.01

delta = 0.04
p0 = (1 - c) * (1 - p1) + delta
p3 = c * (thresholds[3] - thresholds[2])
p2 = 1 - p0 - p1 - p3

priors = np.array([p0, p1, p2, p3])
print(np.sum(priors))

tt = manip_thresh_023(priors, c)
partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c:.6f}")
print(f"True Threshold : {tt:.6f}")
print("Greedy\n------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}\n")
print("Optimal\n-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}\n")
print(f"r (mult)       : {r}")

1.0


,0,1,2,3
Thresholds,0.0000,0.20,0.6000,1.000
Priors,0.0499,0.01,0.5441,0.396


c              : 0.990000
True Threshold : 0.040812
Greedy
------
Partition      : [[0, 1], [2, 3]]
Accuracy loss  : 0.040639

Optimal
-------
Partition      : [[1], [0, 2, 3]]
Accuracy loss  : 0.002453

r (mult)       : 16.564192353108833


In [55]:
c = 0.999
p1 = 1-c

results_8 = {"delta": [], "c": [], "p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r": []}
for delta in np.arange(1e-4, 0.09, 1e-4).round(4):
    # delta = 0.04
    p0 = (1 - c) * (1 - p1) + delta
    p3 = c * (thresholds[3] - thresholds[2])
    p2 = 1 - p0 - p1 - p3

    priors = np.array([p0, p1, p2, p3])

    tt = manip_thresh_023(priors, c)
    partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
    r = approximation_ratio(loss_opt, loss_greedy)

    results_8["delta"].append(delta)
    results_8["c"].append(c)
    results_8["p0"].append(p0)
    results_8["p1"].append(p1)
    results_8["p2"].append(p2)
    results_8["p3"].append(p3)
    results_8["tt"].append(tt)
    results_8["partition_opt"].append(partition_opt)
    results_8["partition_greedy"].append(partition_greedy)
    results_8["loss_opt"].append(loss_opt)
    results_8["loss_greedy"].append(loss_greedy)
    results_8["r"].append(r)

In [56]:
px.scatter(results_8, x="delta", y="r")

In [58]:
delta = results_8["delta"][np.argmax(results_8["r"])]

p0 = (1 - c) * (1 - p1) + delta
p3 = c * (thresholds[3] - thresholds[2])
p2 = 1 - p0 - p1 - p3

priors = np.array([p0, p1, p2, p3])
print(np.sum(priors))

tt = manip_thresh_023(priors, c)
partition_base =[[0],[1],[2],[3]]
partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
loss_base = evaluate_system(X, partition_base, thresholds, priors, tt, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c:.6f}")
print(f"True Threshold : {tt:.6f}\n")
print("Baseline\n--------")
print(f"Partition      : {partition_base}")
print(f"Accuracy loss  : {loss_base}\n")
print("Greedy\n------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy}\n")
print("Optimal\n-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt}\n")
print(f"r (mult)       : {r}")

1.0


,0,1,2,3
Thresholds,0.000000,0.200,0.600000,1.0000
Priors,0.004099,0.001,0.595301,0.3996


c              : 0.999000
True Threshold : 0.003106

Baseline
--------
Partition      : [[0], [1], [2], [3]]
Accuracy loss  : 0.003996003996003996

Greedy
------
Partition      : [[0, 1], [2, 3]]
Accuracy loss  : 0.003992007992007992

Optimal
-------
Partition      : [[1], [0, 2, 3]]
Accuracy loss  : 2.0375624375624385e-05

r (mult)       : 195.9207687781917


In [60]:
results_8 = {"delta": [], "c": [], "p0": [], "p1": [], "p2": [], "p3": [], "tt": [], "partition_opt": [], "partition_greedy": [], "loss_base": [], "loss_opt": [], "loss_greedy": [], "r": []}
for c in tqdm.tqdm(np.arange(0.699, 0.999, 1e-2).round(2)):
    p1 = 1-c
    delta_max = 1e-4
    r_max = -np.inf
    for delta in np.arange(0, 0.09, 1e-3).round(3):
        p0 = (1 - c) * (1 - p1) + delta
        p3 = c * (thresholds[3] - thresholds[2])
        p2 = 1 - p0 - p1 - p3

        priors = np.array([p0, p1, p2, p3])

        tt = manip_thresh_023(priors, c)
        partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
        partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
        r = approximation_ratio(loss_opt, loss_greedy)

        if r > r_max:
            r_max = r
            delta_max = delta

    delta = delta_max

    p0 = (1 - c) * (1 - p1) + delta
    p3 = c * (thresholds[3] - thresholds[2])
    p2 = 1 - p0 - p1 - p3

    priors = np.array([p0, p1, p2, p3])

    tt = manip_thresh_023(priors, c)
    partition_base =[[0],[1],[2],[3]]
    partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
    loss_base = evaluate_system(X, partition_base, thresholds, priors, tt, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
    r = approximation_ratio(loss_opt, loss_greedy)

    
    results_8["delta"].append(delta_max)
    results_8["c"].append(c)
    results_8["p0"].append(p0)
    results_8["p1"].append(p1)
    results_8["p2"].append(p2)
    results_8["p3"].append(p3)
    results_8["tt"].append(tt)
    results_8["partition_opt"].append(partition_opt)
    results_8["partition_greedy"].append(partition_greedy)
    results_8["loss_base"].append(loss_base)
    results_8["loss_opt"].append(loss_opt)
    results_8["loss_greedy"].append(loss_greedy)
    results_8["r"].append(r)
    

100%|██████████| 31/31 [00:21<00:00,  1.46it/s]


In [61]:
pd.DataFrame(results_8)

,delta,c,p0,p1,p2,p3,tt,partition_opt,partition_greedy,loss_base,loss_opt,loss_greedy,r
0,0.001,0.70,0.2110,0.30,0.2090,0.280,0.002041,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.002997,0.001531,0.001531,1.000000
1,0.001,0.71,0.2069,0.29,0.2191,0.284,0.001984,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.001998,0.000993,0.000993,1.000000
2,0.001,0.72,0.2026,0.28,0.2294,0.288,0.001929,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.001998,0.000964,0.000964,1.000000
3,0.001,0.73,0.1981,0.27,0.2399,0.292,0.001877,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.001998,0.000935,0.000935,1.000000
4,0.001,0.74,0.1934,0.26,0.2506,0.296,0.001826,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.001998,0.000906,0.000906,1.000000
5,0.001,0.75,0.1885,0.25,0.2615,0.300,0.001778,"[[1], [0, 2, 3]]","[[1], [0, 2, 3]]",0.001998,0.000876,0.000876,1.000000
6,0.088,0.76,0.2704,0.24,0.1856,0.304,0.152355,"[[1], [0, 2, 3]]","[[1, 3], [0, 2]]",0.152847,0.078013,0.128344,1.645153
7,0.082,0.77,0.2591,0.23,0.2029,0.308,0.138303,"[[1], [0, 2, 3]]","[[1, 3], [0, 2]]",0.138861,0.067917,0.126319,1.859898
8,0.076,0.78,0.2476,0.22,0.2204,0.312,0.124918,"[[1], [0, 2, 3]]","[[1, 3], [0, 2]]",0.124875,0.058392,0.124563,2.133242
9,0.075,0.79,0.2409,0.21,0.2331,0.316,0.120173,"[[1], [0, 2, 3]]","[[1, 3], [0, 2]]",0.120879,0.054504,0.120248,2.206203


In [62]:
px.scatter(results_8, x="c", y="r")

In [ ]:
results_0 = {"c": [], "tt": [], "r": []}
for c in tqdm.tqdm(np.arange(0.1, 1.1, 0.1)):
    p1 = 0.1
    r_max = -np.inf
    delta_max = 1e-4
    for delta in np.arange(1e-4, 0.09, 1e-4).round(4):
        p0 = (1 - c) * (1 - p1) + delta
        p3 = c * (thresholds[3] - thresholds[2])
        p2 = 1 - p0 - p1 - p3

        priors = np.array([p0, p1, p2, p3])

        tt = manip_thresh_023(priors, c)
        partition_opt = find_partitions_optimal(X, thresholds, priors, tt, c)
        partition_greedy = find_partitions_greedy_best(X, thresholds, priors, tt, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, tt, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, tt, c)
        r = approximation_ratio(loss_opt, loss_greedy)

        if r > r_max:
            r_max = r
            delta_max = delta

    delta = delta_max

    p0 = (1 - c) * (1 - p1) + delta
    p3 = c * (thresholds[3] - thresholds[2])
    p2 = 1 - p0 - p1 - p3

    priors = np.array([p0, p1, p2, p3])

    tt = manip_thresh_023(priors, c)

    results_0["c"].append(c)
    results_0["tt"].append(tt)
    results_0["r"].append(r)

In [ ]:
a, b, c, d = [1], [2], [3], [4]

acc_loss_a = evaluate_partition(X, a, thresholds, priors_max, tt, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors_max, tt, c)
acc_loss_c = evaluate_partition(X, c, thresholds, priors_max, tt, c)
acc_loss_d = evaluate_partition(X, d, thresholds, priors_max, tt, c)

# lhs = acc_loss_a * np.sum(priors_max[a]) + acc_loss_b * np.sum(priors_max[b])
# ab = sorted(a + b)

# acc_loss_ab = evaluate_partition(X, ab, thresholds, priors_max, tt, c)
# rhs = acc_loss_ab * np.sum(priors_max[ab])
# gain = lhs - rhs

print(f"    threshold true: {tt:.4f}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"                 c: {c}")
print(f"                 d: {d}")
print(f"conditional loss a: {acc_loss_a:.7f}")
print(f"conditional loss a: {acc_loss_a:.7f}")
print(f"conditional loss c: {acc_loss_c:.7f}")
print(f"conditional loss d: {acc_loss_d:.7f}")
# print(f"            loss a: {acc_loss_a * np.sum(priors_max[a]):.7f}")
# print(f"            loss b: {acc_loss_b * np.sum(priors_max[b]):.7f}")
# print(f"           loss ab: {acc_loss_ab:.7f}")
# print(f"               LHS: {lhs:.7f}")
# print(f"               RHS: {rhs:.7f}")
# print(f"              gain: {gain}")
# print(f"            merge?: {lhs - rhs > -1e-6}\n")

    threshold true: 0.0031
                 c: 0.999
                 a: [1]
                 b: [2]
conditional loss a: 0.0039960
conditional loss b: 0.0039960
            loss a: 0.0000719
            loss b: 0.0015904
           loss ab: 0.0039960
               LHS: 0.0016623
               RHS: 0.0016623
              gain: 0.0
            merge?: True

